# 01 — Exploratory Data Analysis (EDA)Telecom Customer Churn — Stage 2. Read-only profiling, distribution, correlation,and descriptive hypothesis testing of the canonical analysis-ready table.**Input:** `data/processed/clean_customers.csv` (7,043 rows × 43 cols)**Outputs:** figures in `reports/figures/`, findings in `reports/insights/eda_findings.md`> **Churn-rate convention:** `customer_status = 'Joined'` is an acquisition> cohort, not churn. All churn rates use the denominator> `Churned + Stayed = 6,589` (excl. Joined), unless stated otherwise.

In [ ]:
import mathimport osimport numpy as npimport pandas as pdimport matplotlibmatplotlib.use("Agg")import matplotlib.pyplot as pltROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()FIG = os.path.join(ROOT, "reports", "figures")os.makedirs(FIG, exist_ok=True)df = pd.read_csv(os.path.join(ROOT, "data", "processed", "clean_customers.csv"),                 keep_default_na=True)df.shape

## 1. Dataset summary & class balanceConfirm shape, primary key uniqueness, and the Churned/Stayed/Joined split, thencompute the overall churn rate over existing customers only.

In [ ]:
print("rows, cols:", df.shape)print("duplicated rows:", df.duplicated().sum(),      "| duplicated customer_id:", df["customer_id"].duplicated().sum())print(df["customer_status"].value_counts())churned = int((df["customer_status"] == "Churned").sum())stayed  = int((df["customer_status"] == "Stayed").sum())joined  = int((df["customer_status"] == "Joined").sum())existing = churned + stayedprint(f"overall churn rate (excl. Joined): {churned / existing * 100:.2f}%")

## 2. Missing-value profileOnly string columns retain NULLs in the clean table; the boolean/int add-oncolumns were coerced to FALSE/0 during ingestion (001). Verify both facts.

In [ ]:
nulls = df.isna().sum()print("Null counts (non-zero only):")print(nulls[nulls > 0])# confirm coercion: no-internet / no-phone columns are 0/FALSE, not NaNprint("internet_service == False :", int((~df['internet_service']).sum()))print("avg_monthly_gb_download == 0:", int((df['avg_monthly_gb_download'] == 0).sum()))print("phone_service == False    :", int((~df['phone_service']).sum()))print("avg_monthly_long_distance_charges == 0:", int((df['avg_monthly_long_distance_charges'] == 0).sum()))

## 3. Distributions & skewness

In [ ]:
num_cols = ["age", "tenure_months", "monthly_charge", "total_charges",            "total_revenue", "num_referrals", "num_dependents"]df[num_cols].agg(["mean", "median", "std", "skew", "min", "max"]).T.round(3)

## 4. Negative `monthly_charge` investigation120 negative values (all integers −1…−10). Inspect their distribution acrossstatus/contract/tenure to decide credits vs data errors.

In [ ]:
neg = df[df['monthly_charge'] < 0]print("count:", len(neg), "| min:", neg['monthly_charge'].min(),      "| max:", neg['monthly_charge'].max(), "| mean:", round(neg['monthly_charge'].mean(), 2))print("all integer:", bool((neg['monthly_charge'] == neg['monthly_charge'].round()).all()))print("by status:\n", neg['customer_status'].value_counts())print("by contract:\n", neg['contract'].value_counts())print("tenure range:", neg['tenure_months'].min(), "-", neg['tenure_months'].max())print("churn rate of negative rows (excl Joined):",      f"{neg[neg.customer_status != 'Joined']['churn'].mean() * 100:.2f}%")

## 5. Correlations vs churnPoint-biserial (numeric → binary churn) and Cramér's V (categorical → churn).scipy is unavailable, so the chi-square survival function is implementeddirectly via the regularized incomplete gamma.

In [ ]:
def gammp(a, x):    if x < 0.0 or a <= 0.0:        raise ValueError("bad args")    if x < a + 1.0:        ap, su, de = a, 1.0 / a, 1.0 / a        for _ in range(200):            ap += 1.0            de *= x / ap            su += de            if abs(de) < abs(su) * 1e-15:                break        return su * math.exp(-x + a * math.log(x) - math.lgamma(a))    b, c, d, h = x + 1.0 - a, 1e30, 1.0 / (x + 1.0 - a), 1.0 / (x + 1.0 - a)    for i in range(1, 200):        an = -i * (i - a)        b += 2.0        d = an * d + b        if abs(d) < 1e-30: d = 1e-30        c = b + an / c        if abs(c) < 1e-30: c = 1e-30        d = 1.0 / d        de = d * c        h *= de        if abs(de - 1.0) < 1e-15:            break    return 1.0 - math.exp(-x + a * math.log(x) - math.lgamma(a)) * hdef cramers_v(a, b):    m = pd.crosstab(a, b)    n = m.values.sum()    r, c = m.shape    dof = (r - 1) * (c - 1)    if dof <= 0:        return None, None, None, n    exp = (m.values.sum(axis=1, keepdims=True) @ m.values.sum(axis=0, keepdims=True)) / n    chi2 = float(np.sum((m.values - exp) ** 2 / exp))    v = math.sqrt(chi2 / (n * min(r - 1, c - 1)))    p = 1.0 - gammp(dof / 2.0, chi2 / 2.0)    return chi2, v, p, ndef point_biserial(cont, binary):    c = pd.Series(cont).astype(float)    b = pd.Series(binary).astype(float)    m = c.notna() & b.notna()    c, b = c[m], b[m]    return float(np.corrcoef(c, b)[0, 1]) if len(c) > 1 and c.std() > 0 else float("nan")pb_cols = ["age", "num_dependents", "num_referrals", "tenure_months",           "avg_monthly_gb_download", "avg_monthly_long_distance_charges",           "monthly_charge", "total_charges", "total_refunds",           "total_extra_data_charges", "total_long_distance_charges",           "total_revenue", "bundle_count", "contract_commitment"]pb = pd.Series({c: point_biserial(df[c], df["churn"].astype(int)) for c in pb_cols})print("Point-biserial vs churn (sorted):")print(pb.sort_values(key=abs, ascending=False).round(4))

In [ ]:
cat_cols = ["gender", "married", "contract", "payment_method", "internet_type",            "paperless_billing", "phone_service", "multiple_lines",            "internet_service", "offer", "tenure_bin"]rows = []for c in cat_cols:    chi2, v, p, n = cramers_v(df[c], df["churn"])    rows.append((c, round(v, 4), round(chi2, 1), n))cramer = pd.DataFrame(rows, columns=["feature", "cramers_v", "chi2", "n"])print(cramer.sort_values("cramers_v", ascending=False).to_string(index=False))

In [ ]:
heat_cols = ["age", "num_dependents", "num_referrals", "tenure_months",             "avg_monthly_gb_download", "avg_monthly_long_distance_charges",             "monthly_charge", "total_charges", "total_refunds",             "total_revenue", "bundle_count"]corr = df[heat_cols].corr()fig, ax = plt.subplots(figsize=(9, 7))im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)ax.set_xticks(range(len(corr.columns))); ax.set_yticks(range(len(corr.columns)))ax.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=7)ax.set_yticklabels(corr.columns, fontsize=7)for i in range(len(corr.columns)):    for j in range(len(corr.columns)):        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=6)fig.colorbar(im, ax=ax, shrink=0.8)ax.set_title("Numeric feature correlation matrix")fig.tight_layout(); fig.savefig(os.path.join(FIG, "correlation_heatmap.png")); plt.close(fig)

## 6. Hypothesis tests (H1–H6)Churn rates are computed on the existing-customer population(`customer_status != 'Joined'`).

In [ ]:
ex = df[df["customer_status"] != "Joined"]  # churn-rate population (n = 6589)def rate(sub):    return sub["churn"].mean() * 100 if len(sub) else float("nan")def seg_table(col, order=None):    g = ex.groupby(col, dropna=False)    out = [(k, int(len(s)), int(s["churn"].sum()), round(rate(s), 2))           for k, s in g]    if order:        out.sort(key=lambda r: order.index(r[0]) if r[0] in order else 999)    return pd.DataFrame(out, columns=[col, "n", "churned", "churn_rate_pct"])print("H1 — contract");        print(seg_table("contract", ["Month-to-Month", "One Year", "Two Year"]).to_string(index=False))print("\nH2 — internet_type");  print(seg_table("internet_type", ["Fiber Optic", "Cable", "DSL"]).to_string(index=False))print("\nH3 — payment_method");  print(seg_table("payment_method", ["Mailed Check", "Bank Withdrawal", "Credit Card"]).to_string(index=False))print("\nH5 — tenure_bin");      print(seg_table("tenure_bin", ["0-12", "13-24", "25-36", "37-48", "49-60", "60+"]).to_string(index=False))

In [ ]:
# H4 — bundled add-onsfor col in ["premium_tech_support", "online_security", "device_protection_plan"]:    on, off = ex[ex[col]], ex[~ex[col]]    print(f"{col:24s} ON {rate(on):5.2f}% (n={len(on):4d})   OFF {rate(off):5.2f}% (n={len(off):4d})")all3 = ex[ex["premium_tech_support"] & ex["online_security"] & ex["device_protection_plan"]]none3 = ex[~(ex["premium_tech_support"] | ex["online_security"] | ex["device_protection_plan"])]print(f"{'all three combined':24s} churn {rate(all3):5.2f}% (n={len(all3)}) vs none {rate(none3):.2f}% (n={len(none3)})")print(f"{'competitor churn':24s} all3 {(all3['churn_category']=='Competitor').mean()*100:.2f}% vs none3 {(none3['churn_category']=='Competitor').mean()*100:.2f}%")

In [ ]:
# H5 — early tenure velocityle12, gt12 = ex[ex["tenure_months"] <= 12], ex[ex["tenure_months"] > 12]print(f"tenure <= 12: churn {rate(le12):.2f}% (n={len(le12)}) vs > 12: {rate(gt12):.2f}% (n={len(gt12)})")print(f"share of all churn from tenure<=12: {le12['churn'].sum() / ex['churn'].sum() * 100:.2f}%")

In [ ]:
# H6 — churn category & top reasons (churned only)ch = ex[ex["churn"]]print("Churn category:")print((ch["churn_category"].value_counts() / len(ch) * 100).round(1).to_string())print("\nTop churn reasons:")print(ch["churn_reason"].value_counts().head(10).to_string())

## 7. Figures

In [ ]:
def style_ax(ax, title, xlab="", ylab="Churn rate (%)"):    ax.set_title(title); ax.set_xlabel(xlab); ax.set_ylabel(ylab)    ax.grid(axis="y", alpha=0.3)    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)def barfig(name, title, cats, vals, colors):    fig, ax = plt.subplots(figsize=(6, 4))    bars = ax.bar(cats, vals, color=colors)    for b, v in zip(bars, vals):        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.7, f"{v:.1f}%", ha="center")    style_ax(ax, title)    fig.tight_layout(); fig.savefig(os.path.join(FIG, name)); plt.close(fig)h1 = {k: rate(g) for k, g in ex.groupby("contract")}barfig("churn_by_contract.png", "Churn rate by contract type",       ["Month-to-Month", "One Year", "Two Year"],       [h1["Month-to-Month"], h1["One Year"], h1["Two Year"]],       ["#d1495b", "#30638e", "#2a9d8f"])h2 = {("No internet" if pd.isna(k) else k): rate(g)      for k, g in ex.groupby("internet_type", dropna=False)}barfig("churn_by_internet_type.png", "Churn rate by internet type",       ["Fiber Optic", "Cable", "DSL", "No internet"],       [h2["Fiber Optic"], h2["Cable"], h2["DSL"], h2["No internet"]],       ["#d1495b", "#2a9d8f", "#30638e", "#a0a0a0"])h3 = {k: rate(g) for k, g in ex.groupby("payment_method")}barfig("churn_by_payment_method.png", "Churn rate by payment method",       ["Mailed Check", "Bank Withdrawal", "Credit Card"],       [h3["Mailed Check"], h3["Bank Withdrawal"], h3["Credit Card"]],       ["#d1495b", "#2a9d8f", "#30638e"])

In [ ]:
h5 = {k: rate(g) for k, g in ex.groupby("tenure_bin")}tb = ["0-12", "13-24", "25-36", "37-48", "49-60", "60+"]fig, ax = plt.subplots(figsize=(6.5, 4))ax.plot(tb, [h5[t] for t in tb], marker="o", color="#d1495b", linewidth=2)for x, r in zip(tb, [h5[t] for t in tb]):    ax.annotate(f"{r:.1f}%", (x, r), textcoords="offset points", xytext=(0, 8), ha="center")style_ax(ax, "Churn rate by tenure bucket", xlab="Tenure (months)")ax.set_ylim(0, max(h5.values()) * 1.15)fig.tight_layout(); fig.savefig(os.path.join(FIG, "churn_by_tenure_bin.png")); plt.close(fig)fig, ax = plt.subplots(figsize=(7, 4))ax.hist(df["monthly_charge"], bins=60, color="#30638e", alpha=0.85)ax.hist(neg["monthly_charge"], bins=10, color="#d1495b")ax.axvline(0, color="black", linewidth=1)ax.annotate(f"{len(neg)} negative values (min {neg['monthly_charge'].min():.0f})",            xy=(-8, 8), color="#d1495b", fontsize=9, fontweight="bold")ax.set_title("Distribution of monthly_charge (red = negative tail)")ax.set_xlabel("Monthly charge (USD)"); ax.set_ylabel("Count")ax.grid(axis="y", alpha=0.3)ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)fig.tight_layout(); fig.savefig(os.path.join(FIG, "monthly_charge_dist.png")); plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))vc = ch["churn_category"].value_counts()bars = ax.bar(vc.index, vc.values, color="#30638e")for b, c in zip(bars, vc.values):    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 10, str(c), ha="center")ax.set_title("Churn category (churned only)"); ax.set_ylabel("Count")ax.grid(axis="y", alpha=0.3)ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)fig.tight_layout(); fig.savefig(os.path.join(FIG, "churn_category.png")); plt.close(fig)topr = ch["churn_reason"].value_counts().head(12)fig, ax = plt.subplots(figsize=(7, 5))bars = ax.barh(topr.index[::-1], topr.values[::-1], color="#2a9d8f")for b, c in zip(bars, topr.values[::-1]):    ax.text(b.get_width() + 3, b.get_y() + b.get_height()/2, str(c), va="center", fontsize=8)ax.set_title("Top churn reasons (churned only)"); ax.set_xlabel("Count")ax.grid(axis="x", alpha=0.3)ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)fig.tight_layout(); fig.savefig(os.path.join(FIG, "top_churn_reasons.png")); plt.close(fig)print("Figures written to", FIG)